# Topic 21: Transfer Learning: Adapting a Pretrained CNN to Your Own Images

CSCI 3320 · AI Programming in Python · Fall 2026

This notebook is the [Topic 21 lecture note](https://dongchul.kim/csci3320-fall2026/topic/23): the same explanations and pictures,
with every code cell in order. Run the cells from top to bottom (Runtime > Run all works too),
then change things and run them again. Under some code cells, "Output in the note" shows what
that cell printed when the note was made, so you can compare.

**Use a GPU for this one**: Runtime > Change runtime type > T4 GPU (free). Way 2,
fine-tuning, takes roughly 5 to 10 minutes on a T4 and hours on a CPU. Numbers can differ
a little from the note on a different GPU or library version.

In Topic 20 we trained a CNN from random numbers, and on Fashion-MNIST it reached 0.886 with 60,000 labeled images. In real life you rarely have 60,000 labeled images of your own problem. Today we start from a network that **someone else already trained** on 1.2 million photos and adapt it to a new problem with only 5,000. This is called **transfer learning**, and it is how most image models are built in practice.

> **Before this topic.**
>
> **Topic 20**: training a CNN, and the confusion matrix. **Topic 19**: filters and channels. **Topic 14**: the training loop. **Topic 11b** and **Topic 9**: softmax and logistic regression. **Topic 16**: overfitting.

## The idea: keep the body, replace the last layer

A trained CNN has two parts. The **body** is every layer but the last: it turns an image into a short list of numbers that describe it (edges, textures, parts). The **last layer** turns those numbers into one score per class. The body learned things that are useful for almost any photo. The last layer only knows the original classes.

So we copy the body, throw away the last layer, and put a new one there with one output per class of **our** problem:

![Top row: someone else trained ResNet18 on ImageNet: a photo goes through the body, 11,176,512 weights, to 512 numbers, then nn.Linear(512, 1000) gives 1,000 ImageNet scores. Bottom row: we copy the body, and a new nn.Linear(512, 10) with 5,130 weights gives our 10 scores. Way 1 freezes the body and trains only the new layer. Way 2 trains every weight with a small learning rate](https://dongchul.kim/csci3320/notes/topic21/swap-head.svg)

There are two ways to use the copied body:

- **Way 1, frozen**: the body never changes. It is a fixed feature maker, and we train only the new layer.
- **Way 2, fine-tuning**: the body starts from the copied weights, and we train every weight a little.

We will try both, and compare them with Topic 20's CNN trained from scratch on the same images.

## Part 1: the data, and only 5,000 labeled images

**CIFAR-10** has 60,000 small color photos, 32 x 32, in 10 classes: airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck. To act like a real project with few labels, we keep only **5,000** of the 50,000 training images, about 500 per class. The test set stays whole: 10,000 images.

![Left: someone else trained ResNet18 once, on ImageNet, 1.2 million labeled photos in 1,000 classes. Right: today, CIFAR-10. Of its 50,000 training photos we keep 5,000 as your labeled photos and hide the other 45,000. ResNet18 is downloaded and adapted with the 5,000, by way 1 or way 2, and the 10,000 test photos are used only to measure test accuracy](https://dongchul.kim/csci3320/notes/topic21/setup21.png)

We pretend: every one of the 50,000 training photos has a label, but we hide 45,000 of them, so the 5,000 left play **your own small data set**. Later we change 5,000 to anything from 500 to 50,000.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torch.utils.data import TensorDataset, DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
train = torchvision.datasets.CIFAR10("data", train=True, download=True)
test = torchvision.datasets.CIFAR10("data", train=False, download=True)
X_train = torch.tensor(train.data).permute(0, 3, 1, 2).float() / 255   # (50000, 3, 32, 32)
X_test = torch.tensor(test.data).permute(0, 3, 1, 2).float() / 255     # (10000, 3, 32, 32)
y_train, y_test = torch.tensor(train.targets), torch.tensor(test.targets)

keep = torch.randperm(50000)[:5000]          # pretend we only have 5,000 labeled images
X_small, y_small = X_train[keep], y_train[keep]

print(train.data.shape, "->", X_train.shape)
print("we keep:", X_small.shape, "  test:", X_test.shape)
print(train.classes)
print("device:", device)

*Output in the note:*

```text
(50000, 32, 32, 3) -> torch.Size([50000, 3, 32, 32])
we keep: torch.Size([5000, 3, 32, 32])   test: torch.Size([10000, 3, 32, 32])
['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
device: cuda
```

- CIFAR-10 is stored as **height, width, channels**: (50000, 32, 32, 3). `permute(0, 3, 1, 2)` moves the channels to position 1, because `Conv2d` reads **batch, channels, height, width** (Topic 20).
- A color image has **3 channels**: red, green and blue. MNIST had 1.
- `torch.randperm(50000)[:5000]` shuffles the numbers 0 to 49,999 and keeps the first 5,000, so we keep 5,000 random images.
- `device` is `"cuda"` when there is a GPU. Every model and batch goes there with `.to(device)`.

## Part 2: the baseline, Topic 20's CNN from scratch

First, the same program as Topic 20, written once as a function so that only the model changes. `prepare` is something to do to every batch before the model sees it. For now it does nothing.

In [ ]:
def train_and_test(model, X, y, epochs, lr, prepare=lambda x: x):
    model = model.to(device)
    loader = DataLoader(TensorDataset(X, y), batch_size=128, shuffle=True)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = prepare(xb.to(device)), yb.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
    model.eval()
    with torch.no_grad():
        pred = torch.cat([model(prepare(X_test[i:i + 500].to(device))).argmax(1).cpu()
                          for i in range(0, len(X_test), 500)])
    return (pred == y_test).float().mean().item()

Now Topic 20's CNN. Two numbers change, because the images are different: the first `Conv2d` takes **3** channels, and the image is 32 x 32, so after two poolings it is 8 x 8.

In [ ]:
torch.manual_seed(0)
scratch = nn.Sequential(
    nn.Conv2d(3, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),     # 3 channels in: color
    nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(16 * 8 * 8, 10),                                      # 32 -> 16 -> 8
)
print("weights:", sum(p.numel() for p in scratch.parameters()))
acc = train_and_test(scratch, X_small, y_small, epochs=20, lr=0.001)
print(f"from scratch, 5,000 images: test accuracy {acc:.4f}")

*Output in the note:*

```text
weights: 11642
from scratch, 5,000 images: test accuracy 0.4834
```

**0.483**: about half the test photos. With 10 classes, guessing would give 0.10, so it learned something. But CIFAR-10 photos are much harder than digits, and 5,000 images are not many.

## Part 3: a network someone else trained

**ResNet18** is a CNN with 18 layers of weights. Its authors trained it on **ImageNet**: 1.2 million photos in 1,000 classes, from dogs and cats to keyboards and volcanoes. torchvision can download their trained weights in one line:

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights

weights = ResNet18_Weights.DEFAULT
net = resnet18(weights=weights)
print("weights:", sum(p.numel() for p in net.parameters()))
print("trained on ImageNet:", len(weights.meta["categories"]), "classes")
print("its ImageNet validation accuracy:", weights.meta["_metrics"]["ImageNet-1K"]["acc@1"], "%")
print("its last layer:", net.fc)

*Output in the note:*

```text
weights: 11689512
trained on ImageNet: 1000 classes
its ImageNet validation accuracy: 69.758 %
its last layer: Linear(in_features=512, out_features=1000, bias=True)
```

- `weights=ResNet18_Weights.DEFAULT` downloads their trained weights (45 MB) the first time. Without it, `resnet18()` starts from random numbers, like our Topic 20 CNN.
- **11,689,512 weights**, against 11,642 for our small CNN: a thousand times more. Training that from random numbers on 5,000 images would overfit badly. But we do not start from random numbers.
- The last layer, `net.fc`, is `nn.Linear(512, 1000)`: 512 numbers in, one score for each of the 1,000 ImageNet classes out.

### The filters it learned

In Topic 20 we looked at the 8 filters our CNN learned on digits. Here are the 64 filters of ResNet18's first layer, each 7 x 7 and in color:

![64 small color squares in 4 rows. Many are stripes or edges at different angles, some are dots or color blobs](https://dongchul.kim/csci3320/notes/topic21/filters21.png)

Many are **edges and stripes at every angle**, some are color blobs. Nobody wrote them: ImageNet training did. Edges and colors are useful for any photo, cats or trucks or X-rays. That is why the body can be reused.

## Part 4: give it what it expects

ResNet18 was trained on 224 x 224 photos, normalized with the mean and standard deviation of ImageNet's pixels. Our photos must look the same to it:

In [ ]:
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(device)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(device)

def prepare(x):
    x = F.interpolate(x, size=224, mode="bilinear", align_corners=False)   # 32 x 32 -> 224 x 224
    return (x - MEAN) / STD                                                # ImageNet's mean and std

x = X_test[:4].to(device)
print(x.shape, "->", prepare(x).shape)

*Output in the note:*

```text
torch.Size([4, 3, 32, 32]) -> torch.Size([4, 3, 224, 224])
```

- `F.interpolate(x, size=224, mode="bilinear")` stretches each 32 x 32 image to 224 x 224, blending neighbor pixels. It adds no detail, but the sizes now match what the filters learned.
- `(x - MEAN) / STD` standardizes each color channel, as `StandardScaler` did in Topic 9b, with ImageNet's numbers: the network learned with those.
- We call `prepare` on each batch, not on the whole data set: 5,000 images at 224 x 224 would need 3 GB of memory.

### Ask it what it sees, before we change anything

In [ ]:
net = net.to(device).eval()
with torch.no_grad():
    probs = net(prepare(X_test[:4].to(device))).softmax(dim=1)
top = probs.topk(3, dim=1)
for i in range(4):
    guesses = ", ".join(f"{weights.meta['categories'][j]} {p:.2f}"
                        for p, j in zip(top.values[i].tolist(), top.indices[i].tolist()))
    print(f"{test.classes[y_test[i]]:6s} -> {guesses}")

*Output in the note:*

```text
cat    -> sweatshirt 0.09, Windsor tie 0.08, polecat 0.04
ship   -> speedboat 0.66, clog 0.03, airliner 0.03
ship   -> amphibian 0.52, speedboat 0.12, moving van 0.09
airplane -> safety pin 0.18, platypus 0.09, plane 0.06
```

`softmax` turns the 1,000 scores into probabilities (Topic 11b), and `topk(3)` takes the 3 largest. The speedboat for a ship is close. The sweatshirt for a cat is not. Here is its top guess for one test image of each class:

![Ten CIFAR-10 test photos, one per class, with the ImageNet network's top guess under each](https://dongchul.kim/csci3320/notes/topic21/imagenet21.png)

Two reasons it misses. Its 1,000 classes are not our 10: ImageNet has no class called "cat", only kinds of cat, and no "automobile". And a stretched 32 x 32 photo is blurry. So we do not use its last layer. We keep only the body.

## Way 1: a feature maker

Way 1 has two steps. Step 1 turns every image into 512 numbers, once. Step 2 trains only a new last layer on those numbers.

![Step 1, once: 5,000 photos go through the frozen body, which never changes, into a table of 5,000 rows by 512 numbers. Step 2, 20 epochs: the table, 128 rows at a time, goes into the new Linear(512, 10) with 5,130 weights, then to 10 scores and the loss. Learning reaches only the new layer; the body is not in step 2 at all](https://dongchul.kim/csci3320/notes/topic21/way1_21.png)

### Step 1: turn every image into 512 numbers

Replace the last layer with `nn.Identity()`, a layer that passes its input through unchanged. Now the network outputs the **512 numbers** that the last layer used to read. Each image becomes a row of 512 features.

In [ ]:
body = resnet18(weights=weights)
body.fc = nn.Identity()            # no last layer: the output is the 512 numbers before it
body = body.to(device).eval()

@torch.no_grad()
def features(X):
    return torch.cat([body(prepare(X[i:i + 250].to(device))).cpu() for i in range(0, len(X), 250)])

F_small = features(X_small)
F_test = features(X_test)
print(X_small.shape, "->", F_small.shape)
print(X_test.shape, "->", F_test.shape)

*Output in the note:*

```text
torch.Size([5000, 3, 32, 32]) -> torch.Size([5000, 512])
torch.Size([10000, 3, 32, 32]) -> torch.Size([10000, 512])
```

- `body.eval()` and `torch.no_grad()`: we only use the body, we never train it. The body is **frozen**.
- We compute the features **once**. After this, every image is a row of 512 numbers, like a row of a table in Topic 11a.

### Step 2: train only a new last layer

Now train only a new last layer on those rows. This is Topic 14's loop again, on 512 features instead of pixels:

In [ ]:
torch.manual_seed(0)
head = nn.Sequential(nn.Linear(512, 10)).to(device)
loader = DataLoader(TensorDataset(F_small, y_small), batch_size=128, shuffle=True)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(head.parameters(), lr=0.001)
for epoch in range(20):
    for fb, yb in loader:
        fb, yb = fb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = loss_fn(head(fb), yb)
        loss.backward()
        optimizer.step()
with torch.no_grad():
    pred = head(F_test.to(device)).argmax(1).cpu()
print("weights trained:", sum(p.numel() for p in head.parameters()))
print(f"frozen ResNet18 + new last layer, 5,000 images: test accuracy {(pred == y_test).float().mean():.4f}")

*Output in the note:*

```text
weights trained: 5130
frozen ResNet18 + new last layer, 5,000 images: test accuracy 0.8501
```

**0.850**, against 0.483 from scratch, on the same 5,000 images. We trained only **5,130 weights**: $512 \times 10 + 10$.

A single `nn.Linear` with softmax and cross-entropy is softmax regression (Topic 11b), so scikit-learn can do the same job on the same features:

In [ ]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=2000)
clf.fit(F_small.numpy(), y_small.numpy())
print(f"LogisticRegression on the 512 features: test accuracy {clf.score(F_test.numpy(), y_test.numpy()):.4f}")

*Output in the note:*

```text
LogisticRegression on the 512 features: test accuracy 0.8277
```

Close, but not the same: `LogisticRegression` adds an L2 penalty by default (Topic 16), and it trains with a different method. The point stands: **with a good feature maker, a simple classifier is enough.**

## Way 2: fine-tuning

![One step, 20 epochs: 128 photos go through the body, which is trained too with small steps, learning rate 0.0001, then 512 numbers, the new Linear(512, 10), which starts random, and the loss. Learning reaches every weight, 11,181,642 of them](https://dongchul.kim/csci3320/notes/topic21/way2_21.png)

Both ways have the same two parts. The difference is which part learns:

| | the body (ResNet18 without its last layer) | the new last layer, `Linear(512, 10)` |
|---|---|---|
| starts from | ImageNet's weights | random numbers |
| Way 1, frozen | **not trained** | **trained** (5,130 weights) |
| Way 2, fine-tuning | **trained**, small steps (lr 0.0001) | **trained** |

The new last layer always learns: it starts random, and untrained it would get about 1 photo in 10 right.

Now let every weight move. Load the trained network, replace the last layer, and train it all with our function from Part 2. Two things are different from Topic 20: `prepare` makes every batch 224 x 224, and the learning rate is **0.0001**, ten times smaller.

In [ ]:
torch.manual_seed(0)
model = resnet18(weights=weights)
model.fc = nn.Linear(512, 10)          # 1000 ImageNet classes -> our 10
acc = train_and_test(model, X_small, y_small, epochs=20, lr=0.0001, prepare=prepare)
print(f"fine-tuned ResNet18, 5,000 images: test accuracy {acc:.4f}")

*Output in the note:*

```text
fine-tuned ResNet18, 5,000 images: test accuracy 0.8808
```

**0.881**, the best of the three. It took 39 seconds on an A100 GPU. On Colab's free T4 it is slower: expect roughly 5 to 10 minutes.

Why the small learning rate? The copied weights are already good. A large step would wreck them in the first few batches, before the new last layer has learned anything. Small steps **adjust** the body to our photos instead of rewriting it.

### Freezing inside one model

Way 1 above computed the features separately. You can also freeze inside one model, by telling PyTorch not to compute gradients for the copied weights:

In [ ]:
model = resnet18(weights=weights)
for p in model.parameters():
    p.requires_grad = False            # freeze every weight it came with
model.fc = nn.Linear(512, 10)          # a new layer: its weights are trainable
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"trainable {trainable:,} of {total:,}")

*Output in the note:*

```text
trainable 5,130 of 11,181,642
```

- `p.requires_grad = False`: `loss.backward()` finds no gradient for this weight, so the optimizer never moves it.
- The new `nn.Linear` is created **after** the loop, so its weights are trainable, as every new layer is.
- Only **5,130 of 11,181,642** weights train. That is way 1 inside one model.

> **Note.**
>
> If you freeze this way, still call `model.eval()` on the frozen part. ResNet18 contains **BatchNorm** layers, which keep running averages of what they see. In training mode those averages change even when every weight is frozen. Computing the features once with `body.eval()`, as we did, avoids the question.

## How much data do you need?

One run at one size is one number. So we ran all three, 5 seeds each, at 5 training sizes, from 500 images (50 per class) to all 50,000. Up to 5,000 images, every model saw about 100,000 images in total: 200 epochs at 500 images, 20 epochs at 5,000. With all 50,000 images, every model trained for 10 epochs, as in Topic 20.

![Test accuracy on CIFAR-10 against the number of labeled training images, from 500 to 50,000, for three models: the small CNN from scratch, the frozen ResNet18 with a new last layer, and the fine-tuned ResNet18](https://dongchul.kim/csci3320/notes/topic21/curves21_cifar10.png)

The line is the average of 5 seeds, and the band runs from the lowest seed to the highest.

| CIFAR-10, test accuracy | weights trained | 500 | 1,000 | 2,000 | 5,000 | 50,000 |
|---|---|---|---|---|---|---|
| small CNN from scratch | 11,642 | 0.342 | 0.408 | 0.454 | 0.477 | 0.599 |
| Way 1: frozen ResNet18 + new last layer | 5,130 | **0.786** | **0.814** | **0.832** | 0.848 | 0.871 |
| Way 2: fine-tuned ResNet18 + new last layer | 11,181,642 | 0.731 | 0.784 | 0.831 | **0.871** | **0.940** |

Each number is the average of 5 seeds. The best at each size is in bold.

Four things to notice:

- **The copied body helps at every size.** With only 500 images, 50 per class, the frozen ResNet18 gets 0.786. The small CNN needs all 50,000 images to reach 0.599.
- **With few images, frozen wins.** At 500 and 1,000 images, every frozen seed beat every fine-tuned seed. Fine-tuning moves 11 million weights to fit 500 photos: train accuracy 1.000, test 0.731. That is overfitting (Topic 16). The frozen way trains only 5,130 weights, so it has much less room to memorize.
- **With more images, fine-tuning wins.** The two tie at 2,000 (0.832 and 0.831). At 5,000, every fine-tuned seed beat every frozen seed, and with all 50,000 images fine-tuning reaches **0.940** against 0.871. The frozen line flattens: a fixed feature maker can only go so far.
- **The small CNN is underfitting, not overfitting.** At 5,000 images its train accuracy is only 0.542, close to its test accuracy. It cannot even fit its training photos well.

## A different kind of image: Fashion-MNIST

ImageNet is color photos. Fashion-MNIST is small gray pictures of clothes, far from anything in ImageNet. Does the copied body still help? The only change is that a gray image has 1 channel and ResNet18 wants 3, so we copy the one channel three times: `x.repeat(1, 3, 1, 1)`.

![Test accuracy on Fashion-MNIST against the number of labeled training images, from 500 to 60,000, for the same three models](https://dongchul.kim/csci3320/notes/topic21/curves21_fashion.png)

| Fashion-MNIST, test accuracy | weights trained | 500 | 1,000 | 2,000 | 5,000 | 60,000 |
|---|---|---|---|---|---|---|
| small CNN from scratch | 9,098 | 0.779 | 0.812 | 0.830 | 0.835 | 0.884 |
| Way 1: frozen ResNet18 + new last layer | 5,130 | 0.807 | 0.832 | 0.849 | 0.858 | 0.889 |
| Way 2: fine-tuned ResNet18 + new last layer | 11,181,642 | **0.839** | **0.867** | **0.888** | **0.907** | **0.942** |

Each number is the average of 5 seeds. The best at each size is in bold.

- **Fine-tuning wins at every size**, and every fine-tuned seed beat every seed of the other two. With all 60,000 images it reaches **0.942**, against 0.884 for Topic 20's small CNN (the average of these 5 seeds; Topic 20's own runs gave 0.886).
- **The frozen body helps much less here.** Its features were learned on color photos, and small gray clothes look nothing like them. It beats the small CNN by about 0.03 at 500 images, and with all 60,000 the two are tied: 0.889 against 0.884, with overlapping seed ranges.
- So **when your images are unlike ImageNet's, fine-tune**: the body has to adjust to them.

## Which classes improved?

The same 5,000 CIFAR-10 images, seed 0: the CNN from scratch on the left, the fine-tuned ResNet18 on the right. Each row is the true class and adds up to 1,000.

![Two confusion matrices for CIFAR-10. Left, the small CNN from scratch: the diagonal is pale and mistakes are spread everywhere. Right, the fine-tuned ResNet18: the diagonal is dark](https://dongchul.kim/csci3320/notes/topic21/confusion21.png)

These two come from the 5-seed experiment (seed 0), so their accuracies, 0.495 and 0.883, differ a little from the runs above.

- **From scratch, the animals are a mess.** Dog is correct only 253 times of 1,000 and is called cat 223 times. Cat is called frog 206 times. And 275 trucks are called automobile.
- **Fine-tuned, every class is at 745 or above.** Dog goes from 253 to 845. The vehicles are nearly clean: automobile 957, truck 943.
- **The hardest pair is still cat and dog**: 129 cats called dog, 82 dogs called cat.

## The error you will see: a gray image has 1 channel

In [ ]:
gray = torch.rand(8, 1, 28, 28).to(device)          # 8 Fashion-MNIST-sized images
try:
    body(gray)
except RuntimeError as e:
    print("RuntimeError:", str(e))
print("fixed:", body(prepare(gray.repeat(1, 3, 1, 1))).shape)

*Output in the note:*

```text
RuntimeError: Given groups=1, weight of size [64, 3, 7, 7], expected input[8, 1, 28, 28] to have 3 channels, but got 1 channels instead
fixed: torch.Size([8, 512])
```

- The first layer of ResNet18 has weights of size `[64, 3, 7, 7]`: 64 filters, each **3** channels deep.
- A gray batch is `[8, 1, 28, 28]`: **1** channel. So: "expected 3 channels, but got 1".
- The fix: `x.repeat(1, 3, 1, 1)` copies the gray channel into red, green and blue. Then `prepare` makes it 224 x 224.

It is the same family of error as Topic 20's forgotten `unsqueeze`: the channel count is wrong. Print the shape of what you feed the model.

## Which way should you choose?

| your situation | what to try first |
|---|---|
| few labeled images, photos like ImageNet's | **way 1, frozen**: fast, and hard to overfit |
| a few thousand images or more | **way 2, fine-tuning**, with a small learning rate |
| images unlike ImageNet's (gray, medical, satellite) | **way 2**: the body has to adjust |
| no GPU | **way 1**: the features are computed once, then training takes seconds |
| a lot of data and a GPU budget | from scratch is possible, but fine-tuning still usually wins |

Whatever you choose, keep the test set out of every decision, as always.

## Your turn

You want to sort photos of the cafeteria trays into 3 classes: **empty**, **half eaten**, **full**. You have 300 labeled photos.

1. Write the line that replaces ResNet18's last layer for this problem.
2. How many weights does the new layer have?
3. You freeze the body, as in way 1. How many weights train?
4. Which way would you try first, and why?

<details><summary><b>Show the answer</b></summary>

**Answer.**

1. `model.fc = nn.Linear(512, 3)`: the body still gives 512 numbers, and we want 3 scores.
2. $512 \times 3 + 3$ = **1,539**.
3. Only those **1,539**. The other 11,176,512 are frozen.
4. **Way 1, frozen.** 300 photos is very few, about the size where the frozen body beat fine-tuning on CIFAR-10. Train the new layer, look at the confusion matrix, and only then try fine-tuning with a small learning rate, comparing on a validation set (Topic 12).

</details>

## What you learned today

- **Transfer learning**: start from a network someone else trained on a big data set, keep its body, and replace its last layer with one for your classes.
- `resnet18(weights=ResNet18_Weights.DEFAULT)` downloads a network trained on 1.2 million ImageNet photos. `model.fc = nn.Linear(512, 10)` gives it our 10 classes.
- Give it what it expects: **3 channels, 224 x 224, ImageNet's mean and std**.
- **Way 1, frozen**: the body is a feature maker, 512 numbers per image, and only the new layer trains (5,130 weights). **Way 2, fine-tuning**: every weight trains, with a learning rate 10 times smaller.
- On 5,000 CIFAR-10 images: **0.483** from scratch, **0.850** frozen, **0.881** fine-tuned.
- **Few labeled images: frozen wins** (500 and 1,000 CIFAR-10 photos). **More images, or images unlike ImageNet's: fine-tuning wins** (0.940 on all of CIFAR-10, 0.942 on Fashion-MNIST).

## What's next

**Topic 22**, on Thursday: the 512 numbers we used today are an **embedding**: a list of numbers that describes something. We will make embeddings for text too, group things without labels with k-means, and compare a trained model with an LLM used as a classifier.

Back to the [Topic 21 lecture note](https://dongchul.kim/csci3320-fall2026/topic/23).